In [62]:
import time
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder, StandardScaler, MinMaxScaler
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import StratifiedKFold, KFold
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
import itertools
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    roc_auc_score,
    f1_score,
    make_scorer,
    confusion_matrix
)

In [26]:
folder = ""

obesity_df_path = folder + "obesity_shuffled_notscaled.csv"
depression_df_path = folder + "depression_shuffled_notscaled.csv"
congressional_df_train_path = folder + "congressional_df_train_preprocessed.csv"
congresional_df_test_path = folder + "congressional_df_test_preprocessed.csv"
rev_df_train_path = folder + "amazon_review_ID.shuf.lrn.csv"
rev_df_test_path = folder + "amazon_review_ID.shuf.tes.csv"


In [27]:
def train_val_split(train_df):
  nrows = train_df.shape[0]

  train_size = int(0.9 * nrows)

  holdout_train_df = train_df[:train_size]
  holdout_val_df = train_df[train_size:]

  return holdout_train_df, holdout_val_df

In [28]:
def train_test_split(full_df):
  nrows = full_df.shape[0]

  train_size = int(0.8 * nrows)

  train_df = full_df[:train_size]
  test_df = full_df[train_size:]

  return train_df, test_df

In [29]:

obesity_df = pd.read_csv(obesity_df_path)
obesity_df_train, obesity_df_test = train_test_split(obesity_df)
depression_df = pd.read_csv(depression_df_path)
depression_df_train, depression_df_test = train_test_split(depression_df)
congressional_df_train = pd.read_csv(congressional_df_train_path)
congressional_df_test = pd.read_csv(congresional_df_test_path)
rev_df_train = pd.read_csv(rev_df_train_path)
if "ID" in rev_df_train.columns:
  rev_df_train = rev_df_train.drop(columns=["ID"])
rev_df_test = pd.read_csv(rev_df_test_path)

In [30]:
obesity_df_train_holdout, obesity_df_val_holdout = train_val_split(obesity_df_train)
depression_df_train_holdout, depression_df_val_holdout = train_val_split(depression_df_train)
congressional_df_train_holdout, congressional_df_val_holdout = train_val_split(congressional_df_train)
rev_df_train_holdout, rev_df_val_holdout = train_val_split(rev_df_train)

In [31]:
def train_deci_tree_with_grid(df, target_attribute, main_scorer):

  scoring = {
    'accuracy': 'accuracy',
    'precision': make_scorer(precision_score, average='macro', zero_division=0),
    'recall': make_scorer(recall_score, average='macro', zero_division=0),
    'f1': make_scorer(f1_score, average='macro', zero_division=0),
  }

  pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('knn', KNeighborsClassifier())
  ])
  param_grid = {
    'scaler': [StandardScaler(), MinMaxScaler(), None],
    "knn__n_neighbors": range(1, 23, 2),
    "knn__weights": ["uniform", "distance"],
    "knn__metric": ["euclidean", "minkowski", "manhattan"],
    "knn__p": [1, 2],
    "knn__algorithm": ["brute", "auto", "kd_tree", "ball_tree"]
  }

  x = df.loc[:, df.columns != target_attribute]
  y_raw = df[target_attribute]
  le = LabelEncoder()
  y = le.fit_transform(y_raw)

  start = time.perf_counter()

  cv_strategy = KFold(
    n_splits=5,
    shuffle=False,
  )

  grid_search = GridSearchCV(
      estimator=pipe,
      param_grid=param_grid,
      cv=cv_strategy,
      scoring=scoring,
      refit=main_scorer,
      verbose=True,
      n_jobs=-1
  )

  grid_search.fit(x,y)

  results = pd.DataFrame(grid_search.cv_results_)
  elapsed = time.perf_counter() - start
  results["Completion_time"] = elapsed
  print("best accuracy", grid_search.best_score_)
  print(grid_search.best_estimator_)
  print("Time(s): ", elapsed)
  return results, le, grid_search.best_estimator_

In [32]:
def train_deci_tree_with_grid_holdout(df_train, df_val, target_attribute, main_scorer):

  param_grid = {
    'scaler': [StandardScaler(), MinMaxScaler(), None],
    "knn__n_neighbors": range(1, 23, 2),
    "knn__weights": ["uniform", "distance"],
    "knn__metric": ["euclidean", "minkowski", "manhattan"],
    "knn__p": [1, 2],
    "knn__algorithm": ["brute", "auto", "kd_tree", "ball_tree"]
  }

  x_train = df_train.loc[:, df_train.columns != target_attribute]
  y_train_raw = df_train[target_attribute]
  le = LabelEncoder()
  le.fit(pd.concat([df_train[target_attribute], df_val[target_attribute]]))
  y_train = le.transform(y_train_raw)

  x_val = df_val.loc[:, df_val.columns != target_attribute]
  y_val_raw = df_val[target_attribute]
  y_val = le.transform(y_val_raw)

  x_full = pd.concat([x_train, x_val], axis=0)
  y_full = np.concatenate([y_train, y_val])

  start = time.perf_counter()

  param_combinations = list(itertools.product(
      param_grid['scaler'],
      param_grid['knn__n_neighbors'],
      param_grid['knn__weights'],
      param_grid['knn__metric'],
      param_grid['knn__p'],
      param_grid['knn__algorithm'],
  ))

  best_score = 0
  best_model = None
  results_list = []

  for scaler, n_neighbors, weights, metric, p, algorithm in param_combinations:
    steps = []
    if scaler is not None:
        steps.append(('scaler', scaler))
    steps.append(('knn', KNeighborsClassifier(
        n_neighbors=n_neighbors,
        weights=weights,
        metric=metric,
        p=p,
        algorithm=algorithm
    )))
    model = Pipeline(steps)
    
    model.fit(x_train, y_train)
    y_pred = model.predict(x_val)

    result = {
        'param_scaler': type(scaler).__name__ if scaler is not None else None,
        'param_n_neighbors': n_neighbors,
        'param_weights': weights,
        'param_metric': metric,
        'param_p': p,
        'param_algorithm': algorithm,
        'accuracy': accuracy_score(y_val, y_pred),
        'precision': precision_score(y_val, y_pred, average='macro', zero_division=0),
        'recall': recall_score(y_val, y_pred, average='macro', zero_division=0),
        'f1': f1_score(y_val, y_pred, average='macro', zero_division=0)
    }

    results_list.append(result)

    if result[main_scorer] > best_score:
      best_score = result[main_scorer]
      best_model = model

  best_model.fit(x_full, y_full)


  elapsed = time.perf_counter() - start
  results = pd.DataFrame(results_list)
  results['Completion_time'] = elapsed

  print("Best", main_scorer, best_score)
  print("Time(s):", elapsed)
  return results, le, best_model


In [33]:
results_obesity_cv, le_obesity_cv, obesity_best_model_cv = train_deci_tree_with_grid(obesity_df_train, "obesity_level_grouped", "accuracy")
results_depression_cv, le_depression_cv, depression_best_model_cv = train_deci_tree_with_grid(depression_df_train, "depression", "recall")
results_congressional_cv, le_congressional_cv, congressional_best_model_cv = train_deci_tree_with_grid(congressional_df_train, "class", "f1")
results_rev_cv, le_rev_cv, rev_best_model_cv = train_deci_tree_with_grid(rev_df_train, "Class", "f1")

Fitting 5 folds for each of 1584 candidates, totalling 7920 fits
best accuracy 0.9295050304637156
Pipeline(steps=[('scaler', None),
                ('knn',
                 KNeighborsClassifier(algorithm='brute', n_neighbors=1, p=1))])
Time(s):  9.943747662007809
Fitting 5 folds for each of 1584 candidates, totalling 7920 fits
best accuracy 0.8120188794196199
Pipeline(steps=[('scaler', None),
                ('knn',
                 KNeighborsClassifier(algorithm='brute', n_neighbors=21, p=1,
                                      weights='distance'))])
Time(s):  2429.329151007347
Fitting 5 folds for each of 1584 candidates, totalling 7920 fits
best accuracy 0.9432278255197529
Pipeline(steps=[('scaler', MinMaxScaler()),
                ('knn',
                 KNeighborsClassifier(algorithm='ball_tree', n_neighbors=3,
                                      p=1))])
Time(s):  4.6847920110449195
Fitting 5 folds for each of 1584 candidates, totalling 7920 fits
best accuracy 0.343756848763481

In [34]:
%%capture --no-stdout

results_obesity_holdout, le_obesity_holdout, obesity_best_model_holdout = train_deci_tree_with_grid_holdout(obesity_df_train_holdout, obesity_df_val_holdout, "obesity_level_grouped", "accuracy")
results_depression_holdout, le_depression_holdout, depression_best_model_holdout = train_deci_tree_with_grid_holdout(depression_df_train_holdout, depression_df_val_holdout, "depression", "accuracy")
results_congressional_holdout, le_congressional_holdout, congressional_best_model_holdout = train_deci_tree_with_grid_holdout(congressional_df_train_holdout, congressional_df_val_holdout, "class", "f1")
results_rev_holdout, le_rev_holdout, rev_best_model_holdout = train_deci_tree_with_grid_holdout(rev_df_train_holdout, rev_df_val_holdout, "Class", "f1")

Best accuracy 0.9644970414201184
Time(s): 20.724565695039928
Best accuracy 0.8323621694307486
Time(s): 2855.9126576706767
Best f1 0.9536842105263157
Time(s): 12.419020233675838
Best f1 0.33055555555555555
Time(s): 996.0717714922503


In [35]:
def get_top_results(results_df, main_scorer):
  mean_score_metrics = ["mean_test_f1", "mean_test_accuracy", "mean_test_precision", "mean_test_recall"]
  if f"mean_test_{main_scorer}" in mean_score_metrics:
    mean_score_metrics.remove(f"mean_test_{main_scorer}")
  mean_score_metrics.insert(0, f"mean_test_{main_scorer}")
  results_df["combined_rank"] = results_df["rank_test_accuracy"] + results_df["rank_test_precision"] + results_df["rank_test_recall"] + results_df["rank_test_f1"]
  results_df_sorted = results_df.sort_values(by=mean_score_metrics, ascending=False)
  param_cols = [col for col in results_df_sorted.columns if 'param_' in col]
  mean_score_metrics.append("combined_rank")
  relevant_cols = mean_score_metrics + param_cols
  results_df_sorted_relevant = results_df_sorted[relevant_cols]

  return results_df_sorted_relevant


In [36]:
def get_top_results_holdout(results_df, main_scorer):
  mean_score_metrics = ["f1", "accuracy", "precision", "recall"]
  if main_scorer in mean_score_metrics:
    mean_score_metrics.remove(main_scorer)
  mean_score_metrics.insert(0, main_scorer)
  results_df_sorted = results_df.sort_values(by=mean_score_metrics, ascending=False)
  param_cols = [col for col in results_df_sorted.columns if 'param_' in col]
  relevant_cols = mean_score_metrics + param_cols
  results_df_sorted_relevant = results_df_sorted[relevant_cols]

  return results_df_sorted_relevant

In [37]:
processed_results_obesity_cv = get_top_results(results_obesity_cv, "accuracy")
processed_results_depression_cv = get_top_results(results_depression_cv, "recall")
processed_results_congressional_cv = get_top_results(results_congressional_cv, "f1")
processed_results_rev_cv = get_top_results(results_rev_cv, "f1")

In [38]:
processed_results_obesity_holdout = get_top_results_holdout(results_obesity_holdout, "accuracy")
processed_results_depression_holdout = get_top_results_holdout(results_depression_holdout, "recall")
processed_results_congressional_holdout = get_top_results_holdout(results_congressional_holdout, "f1")
processed_results_rev_holdout = get_top_results_holdout(results_rev_holdout, "f1")

In [39]:
processed_results_obesity_cv.head()

,mean_test_accuracy,mean_test_f1,mean_test_precision,mean_test_recall,combined_rank,param_knn__algorithm,param_knn__metric,param_knn__n_neighbors,param_knn__p,param_knn__weights,param_scaler
134,0.929505,0.899797,0.919744,0.896089,4,brute,minkowski,1,1,uniform,None
137,0.929505,0.899797,0.919744,0.896089,4,brute,minkowski,1,1,distance,None
266,0.929505,0.899797,0.919744,0.896089,4,brute,manhattan,1,1,uniform,None
269,0.929505,0.899797,0.919744,0.896089,4,brute,manhattan,1,1,distance,None
272,0.929505,0.899797,0.919744,0.896089,4,brute,manhattan,1,2,uniform,None


In [40]:
processed_results_obesity_holdout.head()

,accuracy,f1,precision,recall,param_scaler,param_n_neighbors,param_weights,param_metric,param_p,param_algorithm
1064,0.964497,0.957996,0.953113,0.964103,None,1,uniform,minkowski,1,brute
1065,0.964497,0.957996,0.953113,0.964103,None,1,uniform,minkowski,1,auto
1066,0.964497,0.957996,0.953113,0.964103,None,1,uniform,minkowski,1,kd_tree
1067,0.964497,0.957996,0.953113,0.964103,None,1,uniform,minkowski,1,ball_tree
1072,0.964497,0.957996,0.953113,0.964103,None,1,uniform,manhattan,1,brute


In [41]:
processed_results_depression_cv.head()

,mean_test_recall,mean_test_f1,mean_test_accuracy,mean_test_precision,combined_rank,param_knn__algorithm,param_knn__metric,param_knn__n_neighbors,param_knn__p,param_knn__weights,param_scaler
257,0.812019,0.816853,0.825233,0.826403,4,brute,minkowski,21,1,distance,None
389,0.812019,0.816853,0.825233,0.826403,4,brute,manhattan,21,1,distance,None
395,0.812019,0.816853,0.825233,0.826403,4,brute,manhattan,21,2,distance,None
653,0.812019,0.816853,0.825233,0.826403,4,auto,minkowski,21,1,distance,None
785,0.812019,0.816853,0.825233,0.826403,4,auto,manhattan,21,1,distance,None


In [42]:
processed_results_depression_holdout.head()

,recall,f1,accuracy,precision,param_scaler,param_n_neighbors,param_weights,param_metric,param_p,param_algorithm
1520,0.820946,0.825154,0.832362,0.832631,None,19,distance,minkowski,1,brute
1521,0.820946,0.825154,0.832362,0.832631,None,19,distance,minkowski,1,auto
1528,0.820946,0.825154,0.832362,0.832631,None,19,distance,manhattan,1,brute
1529,0.820946,0.825154,0.832362,0.832631,None,19,distance,manhattan,1,auto
1532,0.820946,0.825154,0.832362,0.832631,None,19,distance,manhattan,2,brute


In [43]:
processed_results_congressional_cv.head()

,mean_test_f1,mean_test_accuracy,mean_test_precision,mean_test_recall,combined_rank,param_knn__algorithm,param_knn__metric,param_knn__n_neighbors,param_knn__p,param_knn__weights,param_scaler
1333,0.943228,0.944926,0.939145,0.9513,88,ball_tree,minkowski,3,1,uniform,MinMaxScaler()
1334,0.943228,0.944926,0.939145,0.9513,88,ball_tree,minkowski,3,1,uniform,None
1465,0.943228,0.944926,0.939145,0.9513,88,ball_tree,manhattan,3,1,uniform,MinMaxScaler()
1466,0.943228,0.944926,0.939145,0.9513,88,ball_tree,manhattan,3,1,uniform,None
1471,0.943228,0.944926,0.939145,0.9513,88,ball_tree,manhattan,3,2,uniform,MinMaxScaler()


In [44]:
processed_results_congressional_holdout.head()

,f1,accuracy,precision,recall,param_scaler,param_n_neighbors,param_weights,param_metric,param_p,param_algorithm
0,0.953684,0.954545,0.961538,0.95,StandardScaler,1,uniform,euclidean,1,brute
1,0.953684,0.954545,0.961538,0.95,StandardScaler,1,uniform,euclidean,1,auto
2,0.953684,0.954545,0.961538,0.95,StandardScaler,1,uniform,euclidean,1,kd_tree
3,0.953684,0.954545,0.961538,0.95,StandardScaler,1,uniform,euclidean,1,ball_tree
4,0.953684,0.954545,0.961538,0.95,StandardScaler,1,uniform,euclidean,2,brute


In [45]:
processed_results_rev_cv.head()

,mean_test_f1,mean_test_accuracy,mean_test_precision,mean_test_recall,combined_rank,param_knn__algorithm,param_knn__metric,param_knn__n_neighbors,param_knn__p,param_knn__weights,param_scaler
185,0.343757,0.384,0.398997,0.384179,55,brute,minkowski,9,1,distance,None
317,0.343757,0.384,0.398997,0.384179,55,brute,manhattan,9,1,distance,None
323,0.343757,0.384,0.398997,0.384179,55,brute,manhattan,9,2,distance,None
581,0.343757,0.384,0.398997,0.384179,55,auto,minkowski,9,1,distance,None
713,0.343757,0.384,0.398997,0.384179,55,auto,manhattan,9,1,distance,None


In [46]:
processed_results_rev_holdout.head()

,f1,accuracy,precision,recall,param_scaler,param_n_neighbors,param_weights,param_metric,param_p,param_algorithm
1282,0.330556,0.386667,0.371032,0.375,None,9,distance,minkowski,1,kd_tree
1283,0.330556,0.386667,0.371032,0.375,None,9,distance,minkowski,1,ball_tree
1290,0.330556,0.386667,0.371032,0.375,None,9,distance,manhattan,1,kd_tree
1291,0.330556,0.386667,0.371032,0.375,None,9,distance,manhattan,1,ball_tree
1294,0.330556,0.386667,0.371032,0.375,None,9,distance,manhattan,2,kd_tree


In [69]:
def pred_test_data(test_df, model, label_encoder, target_attribute, has_ground_truth):
  x_test = test_df.loc[:, test_df.columns != target_attribute]
  x_test_ids = []
  conf_matrix_df = pd.DataFrame()
  if "ID" in x_test.columns:
    x_test_ids = x_test["ID"]
    x_test = x_test.drop(columns=["ID"])

  if has_ground_truth:
    y_test_raw = test_df[target_attribute]
    y_test = label_encoder.transform(y_test_raw)


  start = time.perf_counter()

  y_pred = model.predict(x_test)

  elapsed = time.perf_counter() - start
  final_results = pd.DataFrame()
  final_results["time"] = [elapsed]
  final_results["parameters"] = [model.get_params()]
  if has_ground_truth:
    final_results["accuracy"] = [accuracy_score(y_test, y_pred)]
    final_results["precision"] = [precision_score(y_test, y_pred, average="macro")]
    final_results["recall"] = [recall_score(y_test, y_pred, average="macro")]
    final_results["f1"] = [f1_score(y_test, y_pred, average="macro")]
    conf_matrix = confusion_matrix(y_test, y_pred)
    class_labels = label_encoder.classes_
    conf_matrix_df = pd.DataFrame(conf_matrix, index=class_labels, columns=class_labels)
    conf_matrix_df.index.name = 'Actual'
    conf_matrix_df.columns.name = 'Predicted'
  final_pred = x_test.copy()
  final_pred["y_pred"] = label_encoder.inverse_transform(y_pred)
  if len(x_test_ids) > 0:
    final_pred["id"] = x_test_ids

  return final_results, final_pred, conf_matrix_df


In [70]:
prediction_results_obesity_cv, y_pred_obesity_cv, conf_matrix_obesity_cv = pred_test_data(obesity_df_test, obesity_best_model_cv, le_obesity_cv, "obesity_level_grouped", True)
prediction_results_depression_cv, y_pred_depression_cv, conf_matrix_depression_cv = pred_test_data(depression_df_test, depression_best_model_cv, le_depression_cv, "depression", True)
prediction_results_congressional_cv, y_pred_congressional_cv, conf_matrix_congressional_cv = pred_test_data(congressional_df_test, congressional_best_model_cv, le_congressional_cv, "class", False)
prediction_results_rev_cv, y_pred_rev_cv, conf_matrix_rev_cv = pred_test_data(rev_df_test, rev_best_model_cv, le_rev_cv, "Class", False)

In [72]:
prediction_results_obesity_holdout, y_pred_obesity_holdout, conf_matrix_obesity_holdout = pred_test_data(obesity_df_test, obesity_best_model_holdout, le_obesity_holdout, "obesity_level_grouped", True)
prediction_results_depression_holdout, y_pred_depression_holdout, conf_matrix_depression_holdout = pred_test_data(depression_df_test, depression_best_model_holdout, le_depression_holdout, "depression", True)
prediction_results_congressional_holdout, y_pred_congressional_holdout, conf_matrix_congressional_holdout = pred_test_data(congressional_df_test, congressional_best_model_holdout, le_congressional_holdout, "class", False)
prediction_results_rev_holdout, y_pred_rev_holdout, conf_matrix_rev_holdout = pred_test_data(rev_df_test, rev_best_model_holdout, le_rev_holdout, "Class", False)

In [50]:
prediction_results_obesity_cv.head()

,time,parameters,accuracy,precision,recall,f1
0,0.007549,"{'memory': None, 'steps': [('scaler', None), (...",0.943262,0.927658,0.903474,0.91039


In [51]:
prediction_results_obesity_holdout.head()

,time,parameters,accuracy,precision,recall,f1
0,0.006946,"{'memory': None, 'steps': [('knn', KNeighborsC...",0.943262,0.927658,0.903474,0.91039


In [52]:
prediction_results_depression_cv.head()

,time,parameters,accuracy,precision,recall,f1
0,0.688552,"{'memory': None, 'steps': [('scaler', None), (...",0.832496,0.830315,0.816918,0.822067


In [53]:
prediction_results_depression_holdout.head()

,time,parameters,accuracy,precision,recall,f1
0,0.774591,"{'memory': None, 'steps': [('knn', KNeighborsC...",0.831062,0.829092,0.815045,0.820376


In [54]:
prediction_results_congressional_cv.head()

,time,parameters
0,0.00281,"{'memory': None, 'steps': [('scaler', MinMaxSc..."


In [55]:
prediction_results_congressional_holdout.head()

,time,parameters
0,0.002429,"{'memory': None, 'steps': [('scaler', Standard..."


In [56]:
prediction_results_rev_cv.head()

,time,parameters
0,3.439906,"{'memory': None, 'steps': [('scaler', None), (..."


In [57]:
prediction_results_rev_holdout.head()

,time,parameters
0,7.91903,"{'memory': None, 'steps': [('knn', KNeighborsC..."


In [58]:
def kaggle_comp_file(pred_df):
  pred_df_final = pred_df[["id", "y_pred"]].rename(columns={"y_pred": "class", "id" : "ID"})
  return pred_df_final


In [59]:
kaggle_submission_congressional_cv = kaggle_comp_file(y_pred_congressional_cv)
kaggle_submission_congressional_holdout = kaggle_comp_file(y_pred_congressional_holdout)

kaggle_submission_congressional_cv.to_csv('congressional_knn_cv_submission_group39.csv', index=False)
kaggle_submission_congressional_holdout.to_csv('congressional_knn_holdout_submission_group39.csv', index=False)

In [60]:
kaggle_submission_rev_cv = kaggle_comp_file(y_pred_rev_cv)
kaggle_submission_rev_holdout = kaggle_comp_file(y_pred_rev_holdout)

kaggle_submission_rev_cv.to_csv('reviews_knn_cv_submission_group39.csv', index=False)
kaggle_submission_rev_holdout.to_csv('reviews_knn_holdout_submission_group39.csv', index=False)

In [86]:
conf_matrix_obesity_cv.head(50)

Predicted,Insufficient_Weight,Normal_Weight,Obesity,Overweight
Actual,,,,
Insufficient_Weight,51,1,0,0
Normal_Weight,7,33,0,7
Obesity,0,0,212,3
Overweight,0,1,5,103


In [87]:
conf_matrix_depression_cv.head(50)

Predicted,0,1
Actual,,
0,1646,579
1,355,2996
